# Unconditional Protein Design


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
from matplotlib.colors import ListedColormap

from scripts.utils import (
    count_amino_acids, load_reversed_trajectory, get_pdb_plddt,
    PLDDT_CONFIDENCE_RANGES, get_plddt_ranges, align_structures,
    get_trajectory_ca_stack,
)

In [ ]:
out_dir = Path("out_unconditional")
out_dir.mkdir(exist_ok=True)

### Structure generation with RFdiffusion

In [ ]:
out_dir_rfdiffusion = out_dir / "out_rfdiffusion"
out_dir_rfdiffusion.mkdir(parents=True, exist_ok=True)

Activate environment:
`conda activate protein-design`

Run RFdiffusion inference in the terminal to generate 10 structures:

```
MDLDIR=./data/generative_models
python ${MDLDIR}/RFdiffusion/scripts/run_inference.py \
    inference.output_prefix=out_unconditional/out_rfdiffusion/design \
    contigmap.contigs=[100-200] \
    inference.num_designs=10
```

Wait until it finishes generating the 10 designs, then run the following cells to visualize the designed structures

In [ ]:
n_designs = 10
n_cols = 5
n_rows = n_designs // n_cols

pdb_paths = [out_dir_rfdiffusion / f"design_{i}.pdb" for i in range(n_designs)]
missing = [str(p) for p in pdb_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing PDB files:\n" + "\n".join(missing))

view = py3Dmol.view(
    width=200*n_cols, height=200*n_rows,
    viewergrid=(n_rows, n_cols), linked=False,
)

for i, pdb_path in enumerate(pdb_paths):
    r, c = divmod(i, n_cols)
    cell = (r, c)

    pdb_text = pdb_path.read_text(encoding="utf-8")
    num_aa = count_amino_acids(pdb_text)
    view.addModel(pdb_text, "pdb", viewer=cell)

    view.setViewStyle({"style": "outline"}, viewer=cell)
    view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=cell)
    view.addLabel(
        f"design_{i} | {num_aa} aa",
        {"position": {"x": 0, "y": 0, "z": 0}, "backgroundColor": "white", "inFront": True,
         "fontColor": "black", "fontSize": 14, "screenOffset": {"x": -50, "y": 100}},
        viewer=cell,
    )
    view.zoomTo(viewer=cell)

view.show()

ELBO estimation for all samples: Estimate how "likely" a chosen design is under the RFdiffusion model (Monte Carlo estimate over sampled diffusion timesteps).

Activate environment:
`conda activate protein-design`

Open the file `scripts/elbo_rfdiffusion.py` and fill in the missing parts of the code to estimate the denoising matching term of the ELBO. 


Then run in the terminal:

```
for N in {0..9}; do
    python scripts/elbo_rfdiffusion.py \
        inference.input_pdb=out_unconditional/out_rfdiffusion/design_${N}.pdb
done
```

This writes `design_${N}.elbo.json` with two scores:
- `denoising_loss`: cheap, uniform-timestep proxy - good for ranking designs against each other
- `weighted_elbo`: monte carlo estimate of the denoising matching term of the ELBO - lower is more likely under the model

The code below plots a scatter plot of the two metrics for each generated structure.

In [ ]:
elbo_paths = sorted(out_dir_rfdiffusion.glob("design_*.elbo.json"))
if not elbo_paths:
    raise FileNotFoundError(
        f"Run scripts/elbo_rfdiffusion.py first; no *.elbo.json files found under {out_dir_rfdiffusion}"
    )

elbo_results = [json.loads(p.read_text(encoding="utf-8")) for p in elbo_paths]
for result in elbo_results:
    name = Path(result["input_pdb"]).stem
    print(f"{name}: denoising_loss={result['denoising_loss']:.4f}  weighted_elbo={result['weighted_elbo']:.2f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter([r["denoising_loss"] for r in elbo_results],
           [r["weighted_elbo"] for r in elbo_results])
for result in elbo_results:
    ax.annotate(Path(result["input_pdb"]).stem, (result["denoising_loss"], result["weighted_elbo"]))
ax.set(xlabel="Denoising loss (uniform-t proxy)", ylabel="Weighted ELBO (nats)",
       title="RFdiffusion ELBO estimates per design")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Pick one design and show the diffusion trajectory by changing n_design as you wish

In [ ]:
n_design = 6  # from 0 to 9

In [ ]:
xt_path = out_dir_rfdiffusion / "traj" / f"design_{n_design}_Xt-1_traj.pdb"
px0_path = out_dir_rfdiffusion / "traj" / f"design_{n_design}_pX0_traj.pdb"

xt_text, xt_n_frames = load_reversed_trajectory(xt_path, hold_final_frames=10)
px0_text, px0_n_frames = load_reversed_trajectory(px0_path, hold_final_frames=10)

if xt_n_frames != px0_n_frames:
    print(f"Warning: Xt-1 has {xt_n_frames} frames, pX0 has {px0_n_frames} frames")

trajectory_viewer = py3Dmol.view(width=1000, height=400, viewergrid=(1, 2), linked=True)
panels = [("Predicted X0", px0_text, (0, 0), 0.5), ("Xt-1", xt_text, (0, 1), 0.8)]

for title, traj_text, cell, zoom in panels:
    trajectory_viewer.addModelsAsFrames(traj_text, "pdb", viewer=cell)
    # Full cartoon
    trajectory_viewer.setStyle({"cartoon": {"color": "spectrum", "opacity": 0.7}}, viewer=cell)
    # CA-only beads
    trajectory_viewer.addStyle(
        {"atom": "CA"}, {"sphere": {"radius": 0.45, "color": "spectrum"}}, viewer=cell,
    )
    # Backbone sticks
    trajectory_viewer.addStyle(
        {"atom": ["N", "CA", "C"]}, {"stick": {"radius": 0.12, "color": "spectrum"}}, viewer=cell,
    )
    trajectory_viewer.addLabel(
        title,
        {"position": {"x": 0, "y": 0, "z": 0}, "backgroundColor": "white", "inFront": True,
         "fontColor": "black", "fontSize": 16, "screenOffset": {"x": -50, "y": 200}},
        viewer=cell,
    )
    trajectory_viewer.zoomTo(viewer=cell)
    trajectory_viewer.zoom(zoom, viewer=cell)

trajectory_viewer.animate({"loop": "forward", "interval": 300})
trajectory_viewer.show()

How many inverse steps does it take before x_t starts looking like a realistic structure?

Confirm this by plotting the distance between x_t and x_0 as a function of t. The code also plots the predicted x_0 (x_0_hat) from x_t. What is different between these two and why?

In [ ]:
# Plot the distance between any point in the trajectory and the final x0
xt_ca_stack = get_trajectory_ca_stack(xt_path.read_text(encoding="utf-8"))
px0_ca_stack = get_trajectory_ca_stack(px0_path.read_text(encoding="utf-8"))

# Frame 0 in the raw files is t=0
final_x0 = xt_ca_stack[0]

# Reverse timesteps (t=T -> t=0)
xt_ca_stack = xt_ca_stack[::-1]
px0_ca_stack = px0_ca_stack[::-1]

n_steps = xt_ca_stack.shape[0]
steps = np.arange(n_steps)

# Per-residue Ca-Ca distance to the final x0
xt_ca_dist = np.linalg.norm(xt_ca_stack - final_x0, axis=2)
px0_ca_dist = np.linalg.norm(px0_ca_stack - final_x0, axis=2)

# Whole-chain Ca-Ca RMSD to the final x0
xt_ca_rmsd = np.sqrt(np.mean(xt_ca_dist**2, axis=1))
px0_ca_rmsd = np.sqrt(np.mean(px0_ca_dist**2, axis=1))

tick_labels = np.arange(n_steps, -1, -10)
tick_positions = np.clip(n_steps - tick_labels, 0, n_steps - 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.3, 1]})

heat_ax = axes[0]
im = heat_ax.imshow(
    xt_ca_dist.T, aspect="auto", origin="lower", cmap="viridis",
    extent=[0, n_steps - 1, 1, xt_ca_dist.shape[1]],
)
cbar = fig.colorbar(im, ax=heat_ax, pad=0.02)
cbar.set_label("Ca distance to final x0 (A)")
heat_ax.set(xlabel="Diffusion generation timestep (noisy -> x0)",
            ylabel="Residue number", title="Xt-1: per-residue Ca distance to final x0",
)
heat_ax.set_xticks(tick_positions)
heat_ax.set_xticklabels(tick_labels)

line_ax = axes[1]
line_ax.plot(steps, xt_ca_rmsd, linewidth=2.5, label="Xt-1 vs final x0")
line_ax.plot(steps, px0_ca_rmsd, linewidth=2.5, label="Predicted x0 vs final x0")
line_ax.set(xlabel="Diffusion generation timestep (noisy -> x0)", 
            ylabel="Ca RMSD (A)", title="Average distance to final x0")
line_ax.set_xticks(tick_positions)
line_ax.set_xticklabels(tick_labels)
line_ax.grid(alpha=0.2)
line_ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Structure-conditioned sequence generation (inverse folding) with ProteinMPNN

Now, let's use Protein MPNN to generate amino acid sequences that fold into the structure we've generated. MPNN is an autoregressive model that uses GNNs to process the input structure and output a sequence one amino acid at the time.

In [ ]:
out_dir_mpnn = out_dir / "out_mpnn"
out_dir_mpnn.mkdir(parents=True, exist_ok=True)

Activate environment:
`conda activate protein-design`

Run ProteinMPNN in the terminal for your chosen structure:

```
N=6
MDLDIR=./data/generative_models
python ${MDLDIR}/ProteinMPNN/protein_mpnn_run.py \
    --pdb_path out_unconditional/out_rfdiffusion/design_${N}.pdb \
    --out_folder out_unconditional/out_mpnn \
    --path_to_model_weights ${MDLDIR}/ProteinMPNN/vanilla_model_weights \
    --num_seq_per_target 5 \
    --sampling_temp 0.1
```

Wait until it finishes generating the 5 sequences, then run the following cells to analyze the generated sequences

In [ ]:
mpnn_fasta = out_dir_mpnn / "seqs" / f"design_{n_design}.fa"

mpnn_records = []
header = None
sequence = []
for line in mpnn_fasta.read_text(encoding="utf-8").splitlines():
    if line.startswith(">"):
        if header is not None:
            mpnn_records.append((header, "".join(sequence)))
        header = line[1:]
        sequence = []
    else:
        sequence.append(line.strip())
if header is not None:
    mpnn_records.append((header, "".join(sequence)))

labels = ["backbone"] + [f"sample {i}" for i in range(1, len(mpnn_records))]
aligned_sequences = [sequence for _, sequence in mpnn_records]
alphabet = sorted(set("".join(aligned_sequences)))
aa_to_index = {aa: i for i, aa in enumerate(alphabet)}
alignment_matrix = np.array([[aa_to_index[aa] for aa in sequence] for sequence in aligned_sequences])

fig, ax = plt.subplots(figsize=(12, 0.4 * len(aligned_sequences) + 1.5))
ax.imshow(alignment_matrix, cmap=ListedColormap(plt.cm.tab20.colors[:len(alphabet)]), aspect="auto")
for row, sequence in enumerate(aligned_sequences):
    for column, aa in enumerate(sequence):
        ax.text(column, row, aa, ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(aligned_sequences[0])))
ax.set_xticklabels([])
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Residue position")
ax.set_title("ProteinMPNN generated sequences")
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

In [ ]:
with open(out_dir_mpnn / f"mpnn_design_{n_design}.fasta", "w", encoding="utf-8") as f:
    for i in range(1, len(aligned_sequences)):
        f.write(f">mpnn_{i}\n")
        f.write(aligned_sequences[i] + "\n")

Do these sequences seem realistic? Use protein BLAST (https://blast.ncbi.nlm.nih.gov/Blast.cgi) to search for similar sequences in existing databases.

### Structure prediction (folding) with ESMFold

Now, we will use ESMFold to predict the most likely 3-D structure of the generated sequences to test if it matches the RFdiffusion structure.

In [ ]:
out_dir_esmfold = out_dir / "out_esmfold"
out_dir_esmfold.mkdir(parents=True, exist_ok=True)

Activate environment:
`conda activate esmfold`

Run ESMFold structure prediction of the 5 ProteinMPNN-generated sequences:

```
MDLDIR=./data/generative_models
HF_HOME=${MDLDIR}/ESMFold/hf-cache \
    python ${MDLDIR}/ESMFold/esmfold.py \
        --fastas_folder out_unconditional/out_mpnn \
        --output_folder out_unconditional/out_esmfold
```

In [ ]:
esmfold_pdbs = sorted(out_dir_esmfold.rglob("*.pdb"))
if not esmfold_pdbs:
    raise FileNotFoundError(f"Run ESMFold first; no PDB files found under {out_dir_esmfold}")

mpnn_sequences = aligned_sequences[1:]
if len(esmfold_pdbs) != len(mpnn_sequences):
    print(
        f"Warning: found {len(esmfold_pdbs)} ESMFold PDBs but {len(mpnn_sequences)} ProteinMPNN sequences."
    )

plddt_by_model = []
for i, pdb_path in enumerate(esmfold_pdbs):
    model_plddt = np.asarray(get_pdb_plddt(pdb_path), dtype=float)
    sequence = mpnn_sequences[i]
    if len(sequence) != len(model_plddt):
        raise ValueError(
            f"{pdb_path.name}: sequence length {len(sequence)} does not match pLDDT length {len(model_plddt)}"
        )
    plddt_by_model.append({
        "sample": i + 1,
        "pdb_path": pdb_path,
        "sequence": sequence,
        "plddt": model_plddt,
        "mean_plddt": float(np.mean(model_plddt)),
    })

best_model = max(plddt_by_model, key=lambda item: item["mean_plddt"])
mpnn_pdb_path = best_model["pdb_path"]
esmfold_sequence = best_model["sequence"]
plddt = best_model["plddt"]

print(f"Selected for downstream visualization: sample {best_model['sample']}")
print(f"PDB: {mpnn_pdb_path}")
print(f"Mean pLDDT: {best_model['mean_plddt']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for plddt_range in PLDDT_CONFIDENCE_RANGES:
    ax.axhspan(
        plddt_range["lower"], plddt_range["upper"],
        color=plddt_range["color"], alpha=0.18,
    )

line_colors = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # green
    "#CC79A7",  # purple
    "#E69F00",  # orange
]
for color, model in zip(line_colors, plddt_by_model):
    positions = np.arange(1, len(model["plddt"]) + 1)
    label = f"{model['sample']}) pLDDT={model['mean_plddt']:.1f}"
    ax.plot(
        positions, model["plddt"], marker="o", markersize=3,
        linewidth=2.5, color=color, label=label,
    )

max_len = len(model["plddt"])
x_ticks = np.unique(np.r_[1, np.arange(10, max_len + 1, 10), max_len])
ax.set_xlim(1, max_len)
ax.set_xticks(x_ticks)
ax.set_xticklabels([str(int(tick)) for tick in x_ticks])

ax.set(xlabel="Residue position", ylabel="pLDDT", ylim=(0, 100),
       title="ESMFold confidence for ProteinMPNN sequences")
ax.grid(axis="y", alpha=0.2)

range_axis = ax.twinx()
range_axis.set_ylim(ax.get_ylim())
range_axis.set_yticks([
    (plddt_range["lower"] + plddt_range["upper"]) / 2
    for plddt_range in PLDDT_CONFIDENCE_RANGES
])
range_axis.set_yticklabels([plddt_range["label"] for plddt_range in PLDDT_CONFIDENCE_RANGES])
range_axis.tick_params(axis="y", length=0, pad=8)

ax.legend(
    bbox_to_anchor=(0.55, -0.3), loc="lower center", ncol=5,
    frameon=False, columnspacing=0.8, labelspacing=0.25,
)
plt.tight_layout()
plt.show()


Is ESMFold able to find a good candidate 3-D structure for the input sequence?

Run the scripts below to visualize the predicted structure of the generated sequence and measure how well it matches the original structure generated by RFdiffusion.

In [ ]:
viewer = py3Dmol.view(width=600, height=400)
viewer.addModel(mpnn_pdb_path.read_text(encoding="utf-8"), "pdb")
viewer.setStyle({}, {"cartoon": {"color": "#0053d6"}})
plddt_ranges = get_plddt_ranges(plddt)
for residue_mask, color in plddt_ranges:
    residues = (np.flatnonzero(residue_mask) + 1).tolist()
    if not residues:
        continue
    selection = {"resi": residues}
    viewer.setStyle(selection, {"cartoon": {"color": color}})
    viewer.addStyle(selection, {"stick": {"radius": 0.1, "color": color}})
viewer.zoomTo()
viewer.show()

In [ ]:

rf_pdb_path = Path(out_dir_rfdiffusion) / f"design_{n_design}.pdb"
rf_pdb_text = rf_pdb_path.read_text(encoding="utf-8")
esmfold_pdb_text = mpnn_pdb_path.read_text(encoding="utf-8")

alignment = align_structures(esmfold_pdb_text, rf_pdb_text)
aligned_esmfold_pdb = alignment["aligned_esmfold_pdb"]
ca_distances = alignment["ca_distances"]
ca_rmsd = alignment["ca_rmsd"]
tm_score = alignment["tm_score"]
length = alignment["length"]

print(f"Aligned Ca residues: {length}")
print(f"Ca RMSD: {ca_rmsd:.3f} A")
print(f"TM-score: {tm_score:.3f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(1, length + 1), ca_distances, color="teal", width=0.8)
ax.axhline(2.0, color="darkorange", linestyle="--", linewidth=1.5)
if len(esmfold_sequence) != length:
    raise ValueError(f"Sequence length {len(esmfold_sequence)} does not match aligned length {length}")
ax.set_xticks(range(1, length + 1))
ax.set_xticklabels(list(esmfold_sequence), fontfamily="monospace")
ax.set_xlim(0.5, length + 0.5)
ax.margins(x=0)
ax.set(xlabel="Amino acid sequence", ylabel="Ca RMSD (A)",
       title="RFdiffusion-ESMFold alignment deviation")
plt.tight_layout()
plt.show()

In [ ]:
view = py3Dmol.view(width=800, height=500)
view.addModel(rf_pdb_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.8}})
view.addModel(aligned_esmfold_pdb, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "#0053d6"}})
plddt_ranges = get_plddt_ranges(plddt)
for residue_mask, color in plddt_ranges:
    residues = (np.flatnonzero(residue_mask) + 1).tolist()
    if not residues:
        continue
    selection = {"model": 1, "resi": residues}
    view.setStyle(selection, {"cartoon": {"color": color}})
    view.addStyle(selection, {"stick": {"radius": 0.1, "color": color}})
label_style = {"backgroundColor": "white", "fontColor": "black", "inFront": True}
view.addLabel("RFdiffusion: gray | ESMFold: pLDDT",
              {**label_style, "screenOffset": {"x": 200, "y": 12}})
view.addLabel(f"RMSD {ca_rmsd:.2f} A | TM {tm_score:.2f}",
              {**label_style, "screenOffset": {"x": 200, "y": -12}})
view.zoomTo()
view.show()

What is your

Wjat 

What is your opinion on the ability of these models to generate realistic structures and sequences?

**Optional**

- Compute self-consistency (RMSD, TM-score) for the 5 ESMFold predicted structures
- Pick the one with the best self-consistency for visualization


Move on to `protein_design_motif_scaffolding.ipynb`